
# SKIRTOR AGN torus: inclination-dependent obscuration and silicate features

Demonstrate how the SKIRTOR clumpy radiative-transfer torus (Stalevski+ 2012, 2016)
reprocesses the hot accretion disc and dust as a function of viewing angle.

**Physical picture:**

- **Face-on (0°, cos_inc=1)**: The hot inner disc dominates. The torus is
  seen from above; torus emission is minimal and the UV/optical big blue bump
  fully visible.

- **Edge-on (90°, cos_inc=0)**: Heavy obscuration. The torus is seen edge-on;
  the disc is completely hidden and the re-radiated MIR/FIR torus emission
  dominates the SED.

The SED transitions continuously as inclination increases, illustrating
the unified AGN model: Seyfert 1s and 2s are the same object viewed at
different angles. The silicate bands (9.7 and 18 μm) transition from
emission to absorption across the inclination range.

**References:**

.. [1] Stalevski, M., Fritz, J., Baes, M., et al. (2012).
   3D radiative transfer modeling of the dusty torus around AGN.
   MNRAS, 420, 2756. arXiv:1109.1286.

.. [2] Stalevski, M., Ricci, C., Ueda, Y., et al. (2016).
   The dust covering factor in AGN — combining the IR torus emission
   with polar dust component. MNRAS, 458, 2288. arXiv:1602.01954.

.. [3] Hao, L., Strauss, M. A., Fan, X., et al. (2007).
   Mid-Infrared Properties of Dust-Obscured Quasars. ApJ, 655, L77.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")
warnings.filterwarnings("ignore", message=".*deprecated.*")

C_AA_PER_S = 2.998e18

# Inclination angles to sweep: 0° (face-on, cos_inc=1) to 90° (edge-on, cos_inc=0)
inclination_deg = np.array([0.0, 15.0, 30.0, 45.0, 60.0, 75.0, 90.0])
cos_inc_values = np.cos(np.radians(inclination_deg))

# Minimal SED model: just the AGN disc + torus
# Negligible host SFH: total mass ~1e-10 Msun, completely subdominant
# to the AGN luminosity below. ``log_sfr`` was the legacy kwarg; current
# ``const`` SFH parametrizes by total mass over [start_gyr, end_gyr]
# instead.
SFH = {"type": "const", "all_params": tengri.FIXED, "log_total_mass": -10.0}
DUST = {"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.0, "tau_bc": 0.0}

ssp = tengri.load_ssp()

# Build two models: one minimal (no dust), one with dust for silicate features
model_agn_only = tengri.SEDModel.build(
    ssp,
    sfh=SFH,
    dust=DUST,
    agn={
        "disc": {"type": "multicolor", "all_params": tengri.FIXED},
        "torus": {"type": "skirtor", "all_params": tengri.FIXED},
        "all_params": tengri.FIXED,
        "log_lbol": 12.0,
        "lum_ratio": 1.0,
    },
    redshift=tengri.Fixed(0.0),
)

model_with_dust = tengri.SEDModel.build(
    ssp,
    sfh={
        "type": "dpl",
        "all_params": tengri.FIXED,
        "tau_gyr": 3.0,
        "log_total_mass": 10.0,
        "alpha": 2.0,
        "beta": 2.5,
    },
    dust={"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.1, "tau_bc": 0.1},
    agn={
        "type": "composable",
        "disc": {"type": "multicolor", "all_params": tengri.FIXED},
        "torus": {"type": "skirtor", "all_params": tengri.FIXED, "tau_skirtor": 7.0},
        "nlr": {"type": "analytic", "all_params": tengri.FIXED},
        "blr": {"type": "none", "all_params": tengri.FIXED},
        "all_params": tengri.FIXED,
        "log_lbol": 12.5,
        "lum_ratio": 1.0,
    },
    redshift=tengri.Fixed(0.05),
)

baseline_agn = dict(model_agn_only.spec.sample(jax.random.PRNGKey(0)))
baseline_dust = dict(model_with_dust.spec.sample(jax.random.PRNGKey(0)))

# Colormap for inclination angle
colors = plt.cm.viridis(np.linspace(0.0, 1.0, len(inclination_deg)))

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16.0, 5.0))

# Panel (a): Full-spectrum inclination sweep (7 angles)
for cos_inc, inc_deg, color in zip(cos_inc_values, inclination_deg, colors):
    params = {**baseline_agn, "agn_cos_inc": np.float64(cos_inc)}
    out = model_agn_only.predict(params)
    wave = np.asarray(model_agn_only.wavelengths)
    sed = np.asarray(out.rest_sed())
    nu_l_nu = C_AA_PER_S / wave * sed

    ax1.loglog(
        wave,
        nu_l_nu,
        color=color,
        lw=1.6,
        label=f"{inc_deg:5.1f}°",
        alpha=0.85,
    )

ax1.set(
    xlabel=r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]",
    ylabel=r"$\nu L_\nu$  [erg s$^{-1}$]",
    xlim=(1e3, 1e6),
    ylim=(1e43, 1e47),
    title="(a) IR SED shape vs cos_inc",
)
legend = ax1.legend(
    title="Inclination",
    frameon=False,
    fontsize=8,
    loc="upper right",
    title_fontsize=8.5,
    ncol=1,
)

for um, _name in [(3.0, "L"), (10.0, "N"), (24.0, "MIPS"), (100.0, "FIR")]:
    ax1.axvline(um * 1.0e4, color="0.85", lw=0.5, alpha=0.5)

# Panel (b): Silicate feature zoom (3–30 μm, 4 angles)
cos_inc_silicate = np.array([1.0, 0.75, 0.5, 0.0])
labels_silicate = [
    r"$\cos \theta = 1.0$ (face-on)",
    r"$\cos \theta = 0.75$",
    r"$\cos \theta = 0.5$",
    r"$\cos \theta = 0.0$ (edge-on)",
]
colors_silicate = ["gold", "orange", "red", "darkblue"]

for cos_inc, color, label in zip(cos_inc_silicate, colors_silicate, labels_silicate):
    params = {**baseline_dust, "agn_cos_inc": jnp.float64(cos_inc)}
    out = model_with_dust.predict(params)

    wave_rest = np.asarray(model_with_dust.wavelengths)
    sed_rest = np.asarray(out.rest_sed())

    # Select 3–30 μm rest-frame window
    mask = (wave_rest >= 3e4) & (wave_rest <= 3e5)
    wave_um = wave_rest[mask] / 1e4
    sed_window = sed_rest[mask]

    ax2.loglog(wave_um, sed_window, color=color, lw=2.0, label=label)

# Mark silicate band centers
silicate_9p7 = 9.7
silicate_18 = 18.0

y_min, y_max = ax2.get_ylim()
ax2.axvline(silicate_9p7, color="gray", linestyle="--", alpha=0.5, linewidth=1.2)
ax2.axvline(silicate_18, color="gray", linestyle="--", alpha=0.5, linewidth=1.2)
ax2.text(
    silicate_9p7,
    y_max * 0.95,
    "9.7 μm",
    fontsize=9,
    ha="center",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7),
)
ax2.text(
    silicate_18,
    y_max * 0.90,
    "18 μm",
    fontsize=9,
    ha="center",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7),
)

ax2.set_xlim(3, 30)
ax2.set_xlabel(r"Rest-frame wavelength $\lambda$ [$\mu$m]", fontsize=11)
ax2.set_ylabel(r"$L_\nu$ [erg s$^{-1}$ Hz$^{-1}$]", fontsize=11)
ax2.set_title("(b) Silicate 9.7 & 18 μm features", fontsize=11)
ax2.legend(fontsize=9, frameon=True, loc="upper right", framealpha=0.95)
ax2.grid(True, alpha=0.3, which="both")

# Panel (c): UV/optical X-ray obscuration vs cos_inc (simplified from cos_inc_sweep)
# Use 5 representative cos_inc values to show the UV → MIR transition
cos_inc_obscuration = np.array([0.95, 0.75, 0.5, 0.25, 0.05])
norm_osc = plt.Normalize(vmin=0.0, vmax=1.0)
cmap_osc = plt.get_cmap("viridis")

for cos_inc in cos_inc_obscuration:
    params = {**baseline_dust, "agn_cos_inc": jnp.float64(cos_inc)}
    out = model_with_dust.predict(params)
    wave = np.asarray(model_with_dust.wavelengths)
    nu = C_AA_PER_S / wave
    nu_l_nu = nu * np.asarray(out.rest_sed())
    ax3.loglog(wave, nu_l_nu, color=cmap_osc(norm_osc(cos_inc)), lw=1.4, alpha=0.85)

ax3.set_xlim(100, 1e6)
ax3.set_ylim(1e40, 1e45)
ax3.set_xlabel(r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]", fontsize=11)
ax3.set_ylabel(r"$\nu L_\nu$  [erg s$^{-1}$]", fontsize=11)
ax3.set_title("(c) UV/optical obscuration", fontsize=11)

sm = plt.cm.ScalarMappable(cmap=cmap_osc, norm=norm_osc)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax3, pad=0.01)
cbar.set_label(r"$\cos \theta_{\rm torus}$", fontsize=10)

fig.tight_layout()
plt.savefig("plot_skirtor_inclination_sweep.png", dpi=150, bbox_inches="tight")
print("Saved plot_skirtor_inclination_sweep.png (consolidated 3-panel figure)")